In [ ]:
/_static/db/webshop.db

# 1. Het relationeel model & sleutelsoorten

:::{admonition} Leerdoelen
:class: tip

In dit hoofdstuk leer je hoe een relationele databank structuur krijgt en hoe tabellen met elkaar verbonden worden.

Na dit hoofdstuk kan je:
- uitleggen wat een primaire sleutel (PK) is
- uitleggen wat een foreign key (FK) is
- relaties herkennen tussen tabellen in een schema
- eenvoudige SQL-voorbeelden interpreteren die PK/FK gebruiken
- uitleggen wat referentiële integriteit betekent
- typische anomalieën herkennen (invoeg-, wijzigings- en verwijderingsanomalie)
:::


In dit hoofdstuk onderzoek je hoe een relationele databank is opgebouwd, welke soorten sleutels erbij horen en waarom referentiële integriteit zo belangrijk is.
We gebruiken opnieuw de database **webshop.db**, die je in de schema-browser kunt bekijken en waarin je SQL-query’s kunt uitvoeren.

## Wat je in dit deel nodig hebt: DB Browser for SQLite

Tot nu toe werkte je volledig op deze website, en dat blijft zo in hoofdstuk 1 tot en met 3. Vanaf hoofdstuk 4 bouw je zelf een databank van nul: je maakt tabellen, vult ze met gegevens, verwijdert ze en maakt ze opnieuw aan met constraints en foreign keys. Daarvoor stap je over naar **DB Browser for SQLite**, een gratis programma op je eigen laptop. Waarom niet gewoon de editor op deze website?

* **Jouw databank wordt een bestand.** In DB Browser is je databank een echt bestand (`voetbal.db`, `cafetaria.db`) dat je bewaart, meeneemt naar huis en indient via Teams. Op deze website blijft wat je met `CREATE TABLE` bouwt alleen in je browser bewaard (je kan het wel als .db-bestand downloaden); het bestand zelf beheren is precies waar het in dit deel om draait.
* **Je ziet je tabellen groeien.** Het tabblad *Browse Data* toont de inhoud van elke tabel terwijl je bouwt; in *Database Structure* zie je je sleutels en constraints terug.
* **Gegevens importeren.** DB Browser leest een CSV-bestand rechtstreeks in als tabel — handig zodra je met echte gegevens werkt.

Zo ervaar je ook wat SQLite eigenlijk is: een databank die in één bestand op je computer zit. Precies dat inzicht heb je nodig in hoofdstuk 6, waar je diezelfde databank naar een server in de cloud brengt.

Installeer het programma nu al, dan ben je klaar tegen hoofdstuk 4:

1. Download [DB Browser for SQLite](https://sqlitebrowser.org/dl/) en installeer het. Op Windows kies je de *Standard installer for 64-bit Windows*, op een Mac het .dmg-bestand.
2. Start het programma één keer op om te controleren of het werkt. Je hoeft geen account aan te maken.
3. Je hoeft nu nog geen databank te openen: in hoofdstuk 4 maak je je eerste databank aan met *New Database*.

:::{note}
Mag je op je laptop geen programma's installeren? Op dezelfde downloadpagina staat een versie zonder installatie (*.zip* of *PortableApp*): uitpakken en starten volstaat. Lukt ook dat niet, zeg het dan aan je leerkracht vóór hoofdstuk 4.
:::

## 1 Tabellen, rijen en kolommen (opfrisser)

In een relationele databank worden gegevens opgeslagen in **tabellen**.
Elke tabel bestaat uit:

* **Kolommen**: de eigenschappen, bv. `first_name`, `unit_price`
* **Rijen**: één concreet record
* **Datatypes**: bepalen welk soort gegevens in een kolom passen

**Voorbeeld (uit webshop.db):**
Bekijk in de schema-browser de tabel `customers`.
Welke kolommen heb je? Welke soort gegevens bevatten ze?


## 2 Primaire sleutels (Primary Keys)

Elke tabel heeft **één kolom** (of een combinatie van kolommen) waarmee je elke rij uniek kunt identificeren.
Dat is de **primaire sleutel (Primary Key — PK)**.

Waarom is dat nodig?

* Om snel rijen terug te vinden
* Om verwarring en dubbele gegevens te vermijden
* Om relaties te leggen met andere tabellen

### Voorbeelden uit webshop.db

* `customers.customer_id`
* `products.product_id`
* `orders.order_id`
* `order_lines` gebruikt een **samengestelde sleutel**:
  `order_id` + `product_id` behoren samen tot één orderregel.

**Voorbeeldquery:**


In [ ]:
SELECT order_id, product_id, quantity 
FROM order_lines 
LIMIT 5;

:::{note}
Waarom zou een orderregel *niet* uniek zijn met alleen `order_id`?
:::

## 3 Unieke sleutels (UNIQUE)

Soms wil je dat een bepaald veld ook uniek is, maar dat het niet de primaire sleutel moet worden.

Voorbeelden die vaak UNIQUE zijn:

* e-mailadres van een klant
* een productcode of SKU
* een gebruikersnaam

In webshop.db is niet alles uniek gemaakt, precies zodat jij later kan oefenen.

**Voorbeeld met UNIQUE:**


In [ ]:
CREATE TABLE new_customers (
    customer_id INTEGER PRIMARY KEY,
    email TEXT UNIQUE,
    first_name TEXT,
    last_name TEXT
);

---- via deze instructies kan je zient dat er een PK en een UNIQUE index is aangemaakt
---- Meer info daarover later
-- PRAGMA table_info(new_customers);
-- PRAGMA index_list(new_customers);
-- PRAGMA index_info('sqlite_autoindex_new_customers_1');

:::{note}
Welke kolommen in webshop.db zouden volgens jou UNIQUE moeten zijn?
(Controleer met een SELECT of er dubbele waarden bestaan.)
:::

## 4 Foreign Keys (FK) – tabellen verbinden

Een **Foreign Key (FK)** is een kolom die verwijst naar de PK van een andere tabel.
Zo ontstaat er een relatie.

### Voorbeelden uit webshop.db

* `orders.customer_id` > verwijst naar `customers.customer_id`
* `order_lines.order_id` > verwijst naar `orders.order_id`
* `order_lines.product_id` > verwijst naar `products.product_id`

Dit geeft structuur: een order kan niet bestaan zonder een klant.

**Voorbeeld:**

```sql
SELECT o.order_id, o.order_date, c.first_name, c.last_name
FROM orders AS o
JOIN customers AS c
    ON o.customer_id = c.customer_id
LIMIT 5;
```

## 5 Referentiële integriteit

Referentiële integriteit betekent:


> Een foreign key moet altijd verwijzen naar een bestaande rij in de andere tabel.

Dus:

* Je mag geen order aanmaken voor een klant die niet bestaat.
* Je mag geen orderregel toevoegen voor een product dat niet bestaat.
* Je mag geen klant verwijderen als die nog bestellingen heeft (tenzij je cascading instelt).

SQLite kan dit afdwingen via FOREIGN KEY-regels (meer hierover bij DDL).

**Check in webshop.db:**


In [ ]:
PRAGMA foreign_keys;

# Als dit `0` geeft, voer dan uit:
# PRAGMA foreign_keys = ON;

## 6 Veelvoorkomende gegevensproblemen (anomalieën)

Slechte of ontbrekende relaties veroorzaken **anomalieën**.
Dit zijn typische problemen die je met goede sleutels kan vermijden.

### 1. Invoeganomalie

Je kunt geen gegevens invoeren omdat er eerst andere gegevens moeten bestaan.

Voorbeeld:
Je wilt een order toevoegen, maar de klant bestaat nog niet.

### 2. Wijzigingsanomalie

Je moet hetzelfde gegeven op meerdere plaatsen aanpassen.

Voorbeeld:
Als een klant verhuist, maar zijn adres staat op:

* de klantentabel
* én mee opgeslagen in elke order

:::{danger}
Nu moet je op meerdere plaatsen aanpassen. Foutgevoelig!
:::

### 3. Verwijderingsanomalie

Door een record te verwijderen verlies je onverwacht andere gegevens.

:::{admonition} Voorbeeld
:class: seealso
Als je de laatste order van een klant verwijdert en je bewaart adresinfo in orders,
dan verlies je ook het adres van die klant.
:::


*Reflectie-opdracht:*
Bekijk de tabel `orders`.
Zit daar informatie in die eigenlijk in `customers` hoort?

## 7 Mini-analyse van webshop.db (wat is goed? wat kan beter?)

Gebruik de schema-browser en bekijk:

* customers
* orders
* order_lines
* products

### Vragen voor analyse (maak notities in je cursus):

1. Welke PK heeft elke tabel?
2. Welke FK’s zie je?
3. Klopt elke relatie logisch met hoe een webshop werkt?
4. Zie je ergens redundantie?
5. Zie je velden die misschien UNIQUE zouden moeten zijn?
6. Welke anomalieën zouden hier kunnen optreden?

:::{Tip} 
Gebruik korte queries om dit te verkennen.
:::

Voorbeeld:

```sql
SELECT email, COUNT(*) 
FROM customers
GROUP BY email
HAVING COUNT(*) > 1;
```

(en kijk of e-mails dubbel voorkomen)


## 8 Vooruitblik — waarom dit belangrijk is voor normalisatie

In het volgende hoofdstuk leer je hoe we tabellen kunnen opsplitsen zodat:

* redundantie verdwijnt
* anomalieën worden vermeden
* structuur logisch en helder blijft
* sleutels een centrale rol spelen (PK/FK)

Dit hoofdstuk is dus de basis voor alles wat met goed databankontwerp te maken heeft.

## Einde van hoofdstuk 1 — Wat je nu zou moeten kunnen

* Het verschil uitleggen tussen PK, FK en UNIQUE
* Kunnen aangeven waarom referentiële integriteit belangrijk is
* Anomalieën herkennen in een slecht ontwerp
* In de schema-browser relaties terugvinden
* Kleine SQL-exploraties uitvoeren om structuur te begrijpen